# GPT SFT Model — Interactive Chat

Load the SFT'd 34M-param GPT checkpoint and chat with it via the project's ChatML-style chat template.

**Note:** this SFT checkpoint is an early-stopped / weak baseline (its SFT data overlapped the pretraining mixture), so outputs will be rough. This notebook is a tinkering harness, not a polished chat model — expect short, sometimes repetitive or off-topic responses.

## 1. Setup

In [1]:
import os

os.environ["HF_HOME"] = "/mnt/ai/data/hf"

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

device: cuda
torch: 2.12.1+cu130


## 2. Config

In [2]:
CKPT_PATH = "/mnt/ai/runs/llm/sft/gpt-384-12-3-6-sft-1B/checkpoints/sft.ckpt"
TOKENIZER_ID = "LiquidAI/LFM2.5-230M"

MODEL_KWARGS = dict(
    block_size=2048,
    n_embd=384,
    n_head=12,
    n_kv_head=3,
    n_layer=6,
    tie_embedding=True,
    mup_base_width=256,
    mup_base_std=0.02,
    mup_input_mult=1.0,
    mup_output_mult=1.0,
)

TEMPERATURE = 0.7
TOP_K = 50
MAX_NEW_TOKENS = 256

## 3. Load tokenizer + model

In [ ]:
from chimera.models import GPT
from chimera.tokenizers import BPETokenizer

tok = BPETokenizer.from_pretrained(TOKENIZER_ID)

model = GPT(vocab_size=tok.vocab_size, **MODEL_KWARGS)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
prefix = "model._orig_mod."
state_dict = {
    k[len(prefix) :]: v for k, v in ckpt["state_dict"].items() if k.startswith(prefix)
}
model.load_state_dict(state_dict)
model.to(device).eval()  # gotcha: chimera does NOT move the model to cuda automatically

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CKPT_PATH}")
print(f"params: {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"vocab_size: {tok.vocab_size}")

loaded /mnt/ai/runs/llm/sft/gpt-384-12-3-6-sft-1B/checkpoints/sft.ckpt
params: 34,030,080 (34.0M)
vocab_size: 64402


## 4. Chat helper

Builds messages, renders them with the canonical chat template (`chimera.data.chat_template`), encodes with `add_special_tokens=False`, and autoregressively samples until `<|im_end|>` or `<|endoftext|>`. Only the newly generated assistant span is decoded and returned.

In [ ]:
from chimera.data import chat_template as ct

im_end_id = tok._tok.token_to_id("<|im_end|>")
eos_id = tok._tok.token_to_id("<|endoftext|>")
stop_ids = {im_end_id, eos_id}
print(f"im_end_id={im_end_id}  eos_id={eos_id}")


@torch.inference_mode()
def chat(
    user_message: str,
    system: str | None = None,
    tools=None,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
) -> str:
    messages = [{"role": "user", "content": user_message}]
    prompt_text = ct.render(
        messages, tools=tools, system=system, add_generation_prompt=True
    )
    prompt_ids = tok._tok.encode(prompt_text, add_special_tokens=False).ids
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    prompt_len = idx.size(1)

    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
        for _ in range(max_new_tokens):
            if idx.size(1) >= model.block_size:
                break
            logits = model(idx)[:, -1, :]
            logits = logits / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
            if nxt.item() in stop_ids:
                break

    new_ids = idx[0, prompt_len:].tolist()
    return tok.decode(new_ids).strip()


print("chat() ready")

im_end_id=7  eos_id=2
chat() ready


## 5. Example chats

In [9]:
print(chat("What is the capital of France?"))

[getOrderInfo(venue_name="France", start_date="2023-05-12", end_date="2024-09-07", end_date="2024-09-08", status="2023-05-07", country="United States")]


In [ ]:
print(
    chat(
        "A store sells apples for $2 each. If I buy 7 apples, how much do I pay in total?"
    )
)

The exchange has been charged to your credit card, and the price difference is $2 + 2 = $2.

Would you like me to proceed with changing your payment method? Please confirm with "yes" to proceed.


In [11]:
print(chat("Write one sentence describing what a neural network is."))

<tool_call>
{"name": "get_definitions", "arguments": {"word": "word", "word": "word"}}
</tool_call>
<tool_call>
{"name": "get_definitions", "arguments": {"word": "word", "word": "word"}}
</tool_call>


## Note on quality

This SFT checkpoint is an early-stopped / weak baseline: the SFT mixture (OpenMathInstruct-style CoT + tool-calling data) overlapped the pretraining mixture, and the run was stopped early. Expect rough, sometimes incoherent or repetitive completions — this notebook is meant for tinkering (swapping in a better checkpoint later, adjusting sampling params, etc.), not as a demonstration of a polished chat model.